**预测**

In [1]:
from ultralytics import YOLO

# 加载模型
model = YOLO("yolo26n.pt")  # 加载官方预训练水平目标检测模型
# model = YOLO("path/to/best.pt")  # 加载自己训练好的自定义检测模型

# 执行图片推理预测
results = model("bus.jpg")  # 对公交车图片做目标检测

# 解析读取检测结果
for result in results:
    xywh = result.boxes.xywh    # 框格式：中心x、中心y、框宽度、框高度（像素绝对坐标）
    xywhn = result.boxes.xywhn  # 归一化xywh，数值范围0~1
    xyxy = result.boxes.xyxy    # 框格式：左上角x、左上角y、右下角x、右下角y（像素绝对坐标）
    xyxyn = result.boxes.xyxyn  # 归一化xyxy，数值范围0~1
    # 获取每个检测框对应的类别名称
    names = [result.names[cls.item()] for cls in result.boxes.cls.int()]
    confs = result.boxes.conf   # 每个目标框的预测置信度（0~1）


image 1/1 C:\Users\wangx\Desktop\python_learn\bus.jpg: 640x480 4 persons, 1 bus, 208.2ms
Speed: 5.6ms preprocess, 208.2ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 480)


**导出**

In [3]:
from ultralytics import YOLO

# 加载模型
model = YOLO("yolo26n.pt")  # 加载官方预训练检测模型
# model = YOLO("path/to/best.pt")  # 加载自己训练完成的自定义模型

# 导出模型为 ONNX 格式
model.export(format="onnx")

Ultralytics 8.4.90  Python-3.11.4 torch-2.12.1+cpu CPU (13th Gen Intel Core i7-1360P)
YOLO26n summary (fused): 122 layers, 2,408,932 parameters, 0 gradients, 5.4 GFLOPs

PyTorch: starting from 'yolo26n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.3 MB)

ONNX: starting export with onnx 1.22.0 opset 20...


C:\Users\wangx\AppData\Roaming\Python\Python311\site-packages\torch\onnx\_internal\torchscript_exporter\symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.94...
ONNX: export success  9.2s, saved as 'yolo26n.onnx' (9.5 MB)

Export complete (10.0s)
Results saved to C:\Users\wangx\Desktop\python_learn\yolo26n.onnx
Predict:         yolo predict task=detect model=yolo26n.onnx imgsz=640 
Validate:        yolo val task=detect model=yolo26n.onnx imgsz=640 data=/home/lq/codes/ultralytics/ultralytics/cfg/datasets/coco.yaml  
Visualize:       https://netron.app


'yolo26n.onnx'

**预测导出图片**

In [8]:
from ultralytics import YOLO

# 加载模型
model = YOLO("yolo26n.pt")  # 加载预训练YOLO26n通用检测模型

# 批量推理：传入图片路径列表，一次性处理多张图片
results = model(["bus.jpg"])  # 返回Results对象列表，一张图对应一个result

# 循环遍历每张图片的推理结果
for result in results:
    boxes = result.boxes          # 水平检测框数据（xyxy、置信度、类别等）
    masks = result.masks           # 实例分割掩码（普通detect模型为None）
    keypoints = result.keypoints   # 人体关键点（pose模型才有，detect模型为None）
    probs = result.probs           # 分类任务概率（检测模型为None）
    obb = result.obb               # 旋转框（obb模型才有，普通检测模型为None）
    
    result.show()                  # 弹出窗口显示画好检测框的图片
    result.save(filename="result.jpg")  # 将绘制完成的图片保存到本地


0: 640x480 4 persons, 1 bus, 150.6ms
Speed: 4.7ms preprocess, 150.6ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 480)


In [12]:
from ultralytics import YOLO

# 加载预训练 YOLO26n 目标检测模型
model = YOLO("yolo26n.pt")

# 指定视频文件路径
source = "path/to/video.mp4"

# 对视频开启流式推理
results = model(source, stream=True)  # 返回 Results 生成器，逐帧输出结果

In [17]:
from ultralytics import YOLO
import cv2

# 加载模型
model = YOLO("yolo26n.pt")
video_path = "dog.mp4"  # 你的原视频路径

# 视频读取
cap = cv2.VideoCapture(video_path)
# 获取视频基础参数
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 创建视频写入器，输出处理后的mp4
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter("output_result.mp4", fourcc, fps, (width, height))

# 流式推理
results = model(video_path, stream=True)

for res in results:
    # 获取绘制完检测框的画面（BGR格式适配opencv）
    frame = res.plot()
    # 写入视频
    out.write(frame)

# 释放资源
cap.release()
out.release()
print("视频导出完成，文件：output_result.mp4")


video 1/1 (frame 1/121) C:\Users\wangx\Desktop\python_learn\dog.mp4: 384x640 1 dog, 149.4ms
video 1/1 (frame 2/121) C:\Users\wangx\Desktop\python_learn\dog.mp4: 384x640 1 dog, 144.1ms
video 1/1 (frame 3/121) C:\Users\wangx\Desktop\python_learn\dog.mp4: 384x640 1 dog, 125.7ms
video 1/1 (frame 4/121) C:\Users\wangx\Desktop\python_learn\dog.mp4: 384x640 1 dog, 120.8ms
video 1/1 (frame 5/121) C:\Users\wangx\Desktop\python_learn\dog.mp4: 384x640 1 dog, 131.4ms
video 1/1 (frame 6/121) C:\Users\wangx\Desktop\python_learn\dog.mp4: 384x640 1 dog, 126.1ms
video 1/1 (frame 7/121) C:\Users\wangx\Desktop\python_learn\dog.mp4: 384x640 1 dog, 94.0ms
video 1/1 (frame 8/121) C:\Users\wangx\Desktop\python_learn\dog.mp4: 384x640 1 dog, 134.3ms
video 1/1 (frame 9/121) C:\Users\wangx\Desktop\python_learn\dog.mp4: 384x640 1 dog, 1 cow, 123.9ms
video 1/1 (frame 10/121) C:\Users\wangx\Desktop\python_learn\dog.mp4: 384x640 1 dog, 116.7ms
video 1/1 (frame 11/121) C:\Users\wangx\Desktop\python_learn\dog.mp4: 38

**---------------------------------------------------------------------
-----------------------------------------------------------**
**图像分割**


In [19]:
#验证

from ultralytics import YOLO

# 加载模型
model = YOLO("yolo26n-seg.pt")  # 官方预训练实例分割模型
# model = YOLO("path/to/best.pt")  # 加载自己训练好的自定义分割模型

# 模型验证评估
metrics = model.val(data="dota8.yaml")  # 无需传参，会复用上次训练缓存的数据集与配置

# 检测框相关指标
metrics.box.map          # 检测框综合mAP（IOU 0.5~0.95均值）
metrics.box.map50        # IOU=0.5时检测框mAP
metrics.box.map75        # IOU=0.75时检测框mAP
metrics.box.maps         # 列表，存放每一类目标各自的mAP50-95值
metrics.box.image_metrics # 字典，按单图存储框检测指标：精确率、召回率、F1、TP、FP、FN

# 实例分割掩码相关指标
metrics.seg.map          # 分割掩码综合mAP（掩码IOU 0.5~0.95均值）
metrics.seg.map50        # 掩码IOU=0.5时mAP
metrics.seg.map75        # 掩码IOU=0.75时mAP
metrics.seg.maps         # 列表，存放每一类掩码各自的mAP50-95值
metrics.seg.image_metrics # 字典，按单图存储分割掩码指标：精确率、召回率、F1、TP、FP、FN

Ultralytics 8.4.90  Python-3.11.4 torch-2.12.1+cpu CPU (13th Gen Intel Core i7-1360P)
YOLO26n-seg summary (fused): 139 layers, 2,722,980 parameters, 0 gradients, 9.1 GFLOPs
val: Fast image access  (ping: 0.40.0 ms, read: 154.216.9 MB/s, size: 97.3 KB)
val: Scanning C:\datasets\dota8\labels\val.cache... 4 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.2it/s 0.8s
                   all          4          8          0          0          0          0          0          0          0          0
            motorcycle          3          4          0          0          0          0          0          0          0          0
                   bus          1          3          0          0          0          0          0          0          0          0
                 bench          1          1          0         

{'P1470__1024__3296___1648.jpg': {'precision': 0.0,
  'recall': 0.0,
  'f1': 0.0,
  'tp': 0,
  'fp': 300,
  'fn': 4},
 'P1571__1024__2976___0.jpg': {'precision': 0.0,
  'recall': 0.0,
  'f1': 0.0,
  'tp': 0,
  'fp': 204,
  'fn': 1},
 'P1580__1024__824___824.jpg': {'precision': 0.0,
  'recall': 0.0,
  'f1': 0.0,
  'tp': 0,
  'fp': 75,
  'fn': 2},
 'P1724__1024__0___824.jpg': {'precision': 0.0,
  'recall': 0.0,
  'f1': 0.0,
  'tp': 0,
  'fp': 300,
  'fn': 1}}

In [22]:
from ultralytics import YOLO
import cv2
import numpy as np

# 加载实例分割模型
model = YOLO("yolo26n-seg.pt")

# 图片推理
results = model("bus.jpg")

# 遍历结果
for result in results:
    # 分割掩码相关数据
    xy = result.masks.xy      # 掩码轮廓像素坐标
    xyn = result.masks.xyn    # 归一化轮廓坐标
    masks = result.masks.data # 二值掩码张量 (N,H,W)
    
    # 1. 获取绘制好掩码、框、类别标签的原图
    img = result.plot()
    # 转换RGB转BGR适配OpenCV显示
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    # 2. OpenCV弹窗显示图片
    cv2.imshow("Segment Result", img_bgr)
    print("===== 分割掩码信息 =====")
    print(f"目标数量：{len(xy)}")
    for idx, contour in enumerate(xy):
        print(f"第{idx+1}个物体轮廓点数量：{len(contour)}")
    
    # 3. 可选：单独提取并显示单个掩码
    if len(masks) > 0:
        first_mask = masks[0].cpu().numpy() * 255
        cv2.imshow("First Mask", first_mask)

# 按任意键关闭窗口
cv2.waitKey(0)
cv2.destroyAllWindows()


image 1/1 C:\Users\wangx\Desktop\python_learn\bus.jpg: 640x480 4 persons, 1 bus, 182.4ms
Speed: 4.5ms preprocess, 182.4ms inference, 6.8ms postprocess per image at shape (1, 3, 640, 480)
===== 分割掩码信息 =====
目标数量：5
第1个物体轮廓点数量：367
第2个物体轮廓点数量：678
第3个物体轮廓点数量：242
第4个物体轮廓点数量：184
第5个物体轮廓点数量：137


**语义分割**

In [24]:
from ultralytics import YOLO
import cv2
import numpy as np

# 加载语义分割模型
model = YOLO("yolo26n-sem.pt")
# model = YOLO("path/to/best.pt")

# 执行图片推理
results = model("bus.jpg")

# 遍历推理结果
for result in results:
    # 获取语义分割掩码，单通道(H,W)，像素值为类别ID

    semantic_mask = result.semantic_mask.data

    # 1. 绘制带彩色语义分割标注的原图
    draw_img = result.plot()
    # RGB转BGR适配OpenCV
    draw_img_bgr = cv2.cvtColor(draw_img, cv2.COLOR_RGB2BGR)
    cv2.imshow("语义分割效果图", draw_img_bgr)

    # 2. 单独可视化类别掩码灰度图
    mask_np = semantic_mask.cpu().numpy().astype(np.uint8)
    cv2.imshow("类别ID掩码图", mask_np)

    # 打印掩码信息
    print(f"语义掩码尺寸 H×W：{mask_np.shape}")
    print(f"画面中出现的所有类别ID：{np.unique(mask_np)}")

# 等待按键，关闭所有窗口
cv2.waitKey(0)
cv2.destroyAllWindows()


image 1/1 C:\Users\wangx\Desktop\python_learn\bus.jpg: 1024x768 212.8ms
Speed: 16.0ms preprocess, 212.8ms inference, 212.2ms postprocess per image at shape (1, 3, 1024, 768)
语义掩码尺寸 H×W：(1080, 810)
画面中出现的所有类别ID：[ 0  1  2  4  5  7  8 11 13 14 16]


**图像分类**

In [26]:
from ultralytics import YOLO
import cv2

# 加载图像分类模型
model = YOLO("yolo26n-cls.pt")
# model = YOLO("path/to/best.pt")

# 图片推理
results = model("bus.jpg")

# 遍历分类结果
for result in results:
    top1 = result.probs.top1               # 置信度最高类别ID
    top1_conf = result.probs.top1conf      # 对应置信度
    top1_name = result.names[top1]         # 类别名称

    # 控制台打印分类信息
    print("========分类预测结果========")
    print(f"最优类别ID：{top1}")
    print(f"类别名称：{top1_name}")
    print(f"置信度：{float(top1_conf):.4f}")

    # 绘制预测文字并使用cv2展示图片
    img_rgb = result.plot()
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    cv2.imshow("图像分类结果", img_bgr)

# 按任意按键关闭图片窗口
cv2.waitKey(0)
cv2.destroyAllWindows()


image 1/1 C:\Users\wangx\Desktop\python_learn\bus.jpg: 224x224 minibus 0.52, police_van 0.37, trolleybus 0.10, minivan 0.01, ambulance 0.00, 22.1ms
Speed: 10.3ms preprocess, 22.1ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)
========分类预测结果========
最优类别ID：654
类别名称：minibus
置信度：0.5165


**姿势评估**

In [27]:
from ultralytics import YOLO
import cv2

# 加载人体姿态关键点模型
model = YOLO("yolo26n-pose.pt")
# model = YOLO("path/to/best.pt")

# 图片推理预测
results = model("bus.jpg")

# 遍历结果解析关键点数据
for result in results:
    xy = result.keypoints.xy        # 关键点像素坐标
    xyn = result.keypoints.xyn      # 归一化关键点坐标
    kpts = result.keypoints.data    # 完整关键点：x,y,可见度
    
    # 打印检测信息
    print("====人体姿态关键点信息====")
    print(f"检测到人数：{xy.shape[0]}")
    print(f"关键点数据维度：{kpts.shape}")

    # 绘制带人体骨架、关键点的图片
    img_rgb = result.plot()
    # RGB转BGR适配OpenCV显示
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    cv2.imshow("人体姿态检测效果图", img_bgr)

# 等待按键关闭窗口
cv2.waitKey(0)
cv2.destroyAllWindows()


image 1/1 C:\Users\wangx\Desktop\python_learn\bus.jpg: 640x480 4 persons, 169.4ms
Speed: 4.9ms preprocess, 169.4ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 480)
====人体姿态关键点信息====
检测到人数：4
关键点数据维度：torch.Size([4, 17, 3])


**obb检测**

In [28]:
from ultralytics import YOLO
import cv2

# 加载旋转框OBB检测模型
model = YOLO("yolo26n-obb.pt")
# model = YOLO("path/to/best.pt")

# 图片推理预测
results = model("https://ultralytics.com/images/boats.jpg")

# 遍历每张图片的检测结果
for result in results:
    # 旋转框参数：中心x,中心y,宽,高,旋转角度（弧度）
    xywhr = result.obb.xywhr
    # 四边形四点像素坐标，格式 [x1,y1,x2,y2,x3,y3,x4,y4]
    xyxyxyxy = result.obb.xyxyxyxy
    # 每个旋转框对应的类别名称
    names = [result.names[cls.item()] for cls in result.obb.cls.int()]
    # 每个框的置信度分数
    confs = result.obb.conf

    # 控制台输出检测信息
    print("==== OBB旋转框检测信息 ====")
    print(f"图片中检测到目标总数：{len(names)}")
    for idx, cls_name in enumerate(names):
        print(f"目标{idx+1} | 类别：{cls_name} | 置信度：{float(confs[idx]):.4f}")

    # 绘制带旋转框、类别、置信度的标注图
    img_rgb = result.plot()
    # RGB转BGR，适配OpenCV显示，防止颜色偏色
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    cv2.imshow("OBB旋转目标检测效果图", img_bgr)

# 按键盘任意键关闭图片窗口
cv2.waitKey(0)
cv2.destroyAllWindows()


Found https://ultralytics.com/images/boats.jpg locally at boats.jpg
image 1/1 C:\Users\wangx\Desktop\python_learn\boats.jpg: 576x1024 170 ships, 3 harbors, 220.7ms
Speed: 9.1ms preprocess, 220.7ms inference, 0.6ms postprocess per image at shape (1, 3, 576, 1024)
==== OBB旋转框检测信息 ====
图片中检测到目标总数：173
目标1 | 类别：ship | 置信度：0.9430
目标2 | 类别：ship | 置信度：0.9216
目标3 | 类别：ship | 置信度：0.9203
目标4 | 类别：ship | 置信度：0.9157
目标5 | 类别：ship | 置信度：0.9153
目标6 | 类别：ship | 置信度：0.9150
目标7 | 类别：ship | 置信度：0.9141
目标8 | 类别：ship | 置信度：0.9134
目标9 | 类别：ship | 置信度：0.9121
目标10 | 类别：ship | 置信度：0.9118
目标11 | 类别：ship | 置信度：0.9111
目标12 | 类别：ship | 置信度：0.9104
目标13 | 类别：ship | 置信度：0.9104
目标14 | 类别：ship | 置信度：0.9101
目标15 | 类别：ship | 置信度：0.9092
目标16 | 类别：ship | 置信度：0.9092
目标17 | 类别：ship | 置信度：0.9074
目标18 | 类别：ship | 置信度：0.9070
目标19 | 类别：ship | 置信度：0.9057
目标20 | 类别：ship | 置信度：0.9053
目标21 | 类别：ship | 置信度：0.9049
目标22 | 类别：ship | 置信度：0.9047
目标23 | 类别：ship | 置信度：0.9042
目标24 | 类别：ship | 置信度：0.9035
目标25 | 类别：ship | 置信度：0.9034
目标26 | 类别